# Plot both magnetometer and waves data between a specified start and end date.

In [ ]:
import os
import re
import glob
import time
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt

from matplotlib.ticker import AutoMinorLocator

from JunoPy.mag import MagData
from JunoPy.waves import WavesData
from JunoPy.library import find_ticks_between

In [ ]:
## 1. Select the file(s) to plot.

# '2017-236T20:21:15 2017-236T20:21:38 NP',
# '2017-338T01:49:57 2017-338T01:51:00 PN',
# '2019-089T20:16:40 2019-089T20:17:00 NP',
# '2018-353T04:04:31 2018-353T04:04:50 PN',    # Change times 
# '2018-353T13:12:15 2018-353T13:12:40 NP',
# '2019-089T20:35:40 2019-089T20:36:36'
start = dt.datetime.strptime('2017-338T01:49:57', '%Y-%jT%H:%M:%S') - dt.timedelta(minutes=10)
end   = dt.datetime.strptime('2017-338T01:51:00', '%Y-%jT%H:%M:%S') + dt.timedelta(minutes=10)

magfile = glob.glob('data/mag/*{:s}ss*.hdf5'.format(start.strftime('%Y%j')))[0]
magdata = MagData(filename=magfile)

wavesfiles = glob.glob('data/waves-survey/*{:s}*.CSV'.format(start.strftime('%Y%j')))[:2]
if '_E_' in wavesfiles[0]:
    wavesdataE = WavesData(filename=wavesfiles[0])
    wavesdataB = WavesData(filename=wavesfiles[1])
else:
    wavesdataE = WavesData(filename=wavesfiles[1])
    wavesdataB = WavesData(filename=wavesfiles[0])
print(wavesdataE)

In [ ]:
def plot_data(magdata, wavesdataE, wavesdataB, start, end):
    """Plot data between start and end."""
    
    # Slice the data.
    magdata.df = magdata.df.loc[start:end]
    wavesdataE.df = wavesdataE.df.loc[start:end]
    wavesdataB.df = wavesdataB.df.loc[start:end]
    
    # Find the ticks.
    timearray, tickstring = find_ticks_between(start, end)
    
    # Create the figure.
    fig, axs = plt.subplots(5, 1, 
                            gridspec_kw={'hspace': 0, 'right':0.9, 'top':0.9}, 
                            figsize=(9, 11), 
                            dpi=300)
    
    axs[:4] = magdata.plot(vars=['Br', 'Btheta', 'Bphi', 'Bmag'], axs=axs[:4])
    axs[4] = wavesdataE.plot(axs[4], magdata=magdata)
    
    for a in axs:
        # a.set_xlim([timearray[0], timearray[-1]])
        a.set_xticks(timearray)
        a.set_xticklabels(tickstring)
        a.label_outer()
        
    axs[4].grid(zorder=2, which='both', lw=0.5)
        
    for a in axs[:]:
        a.axvspan(start + dt.timedelta(minutes=30), 
                  end - dt.timedelta(minutes=30), alpha=0.2, color='red')
            
    a.tick_params(axis='x', width=2, rotation=0)
    a.xaxis.label.set_visible(False)
    
    plt.suptitle(dt.datetime.strftime(start, '%Y-%m-%d') + \
                 ', DOY={:s}'.format(dt.datetime.strftime(start, '%j')), y=0.92)
    
    fig.savefig('out_waves_selected/' + dt.datetime.strftime(start, '%Y-%jT%H:%M') + '.png')
    # plt.close(fig)
    
plot_data(magdata, wavesdataE, wavesdataB, start, end)

In [ ]:
def plot_waves_data(wavesdataE, start, end):
    """Plot data between start and end."""
    
    # Slice the data.
    wavesdataE.df = wavesdataE.df.loc[start:end]
    
    # Find the ticks.
    timearray, tickstring = find_ticks_between(start, end)
    
    # Create the figure.
    fig, axs = plt.subplots(1, 1, 
                            gridspec_kw={'hspace': 0, 'right':0.9, 'top':0.9}, 
                            figsize=(9, 4.5), 
                            dpi=300)
    
    axs = wavesdataE.plot(axs)
    
    # a.set_xlim([timearray[0], timearray[-1]])
    axs.set_xticks(timearray)
    axs.set_xticklabels(tickstring)
    axs.label_outer()
        
    axs.grid(zorder=2, which='both', lw=0.5)
        

    axs.axvspan(start + dt.timedelta(minutes=30), 
                end - dt.timedelta(minutes=30), alpha=0.2, color='red')
            
    axs.tick_params(axis='x', width=2, rotation=0)
    axs.xaxis.label.set_visible(False)
    
    plt.suptitle(dt.datetime.strftime(start, '%Y-%m-%d') + \
                 ', DOY={:s}'.format(dt.datetime.strftime(start, '%j')), y=0.94)
    
    fig.savefig('out_waves_selected/' + dt.datetime.strftime(start, '%Y-%jT%H:%M_waves') + '.png')
    # plt.close(fig)
    
plot_waves_data(wavesdataE, start, end)

In [ ]:
def plot_waves_data(wavesdataE, start, end):
    """Plot data between start and end."""
    
    # Slice the data.
    # wavesdataE.df = wavesdataE.df.loc[start:end]
    
    # Find the ticks.
    timearray, tickstring = find_ticks_between(start, end)
    
    # Create the figure.
    fig, axs = plt.subplots(1, 1, 
                            gridspec_kw={'hspace': 0, 'right':0.9, 'top':0.9}, 
                            figsize=(9, 4.5), 
                            dpi=300)
    
    axs = wavesdataE.plot(axs, freqlimit=True, vmin=1e-14, vmax=1e-10)
    
    # a.set_xlim([timearray[0], timearray[-1]])
    axs.set_xticks(timearray)
    axs.set_xticklabels(tickstring)
    axs.label_outer()
        
    axs.grid(zorder=2, which='both', lw=0.5)
        

#     axs.axvspan(start + dt.timedelta(minutes=30), 
#                 end - dt.timedelta(minutes=30), alpha=0.2, color='red')
            
    axs.tick_params(axis='x', width=2, rotation=0)
    axs.xaxis.label.set_visible(False)
    
    plt.suptitle(dt.datetime.strftime(start, '%Y-%m-%d') + \
                 ', DOY={:s}'.format(dt.datetime.strftime(start, '%j')), y=0.94)
    
    fig.savefig('out_waves_selected/' + dt.datetime.strftime(start, '%Y-%jT%H:%M_waves') + '.png')
    # plt.close(fig)
    
plot_waves_data(wavesdataE, start, end)

In [ ]:
# Extra stuff

In [ ]:
omega = np.array([500, 1000, 2000, 2500, 5e4]) * 2 * np.pi # Hz

In [ ]:
from scipy.constants import epsilon_0, m_e, e, m_p, c, mu_0

In [ ]:
n_e = omega**2 * epsilon_0 * m_e / e**2 / 1e6
n_e

In [ ]:
d_i = c / (omega * np.sqrt(m_e / m_p / 16.6)) / 1e3
d_i

In [ ]:
19 * 450

In [ ]:
62 * 450

In [ ]:
B = 5 * 1e-9
rho = 0.012 * 1e6 * 16.6 * m_p
vA = B / np.sqrt(mu_0 * rho)
print(vA / 1000)

In [ ]:
62 * 244